#Window Functions

# 🪟 Spark 16: Window Functions

| | |
|---|---|
| **Topic** | Window Functions |
| **Source** | Udemy course + own practice |
| **Practice in** | PySpark · Spark SQL · Standard SQL |


## 1️⃣ What is a Window Function?

> **Definition:** A window function calculates a value across a group of related rows **while keeping every row**.

🎤 **Interview answer:**
*"GROUP BY collapses rows; a window function doesn't. It adds a calculated column to each row, like a rank within a department or a running total."*

## 2️⃣ GROUP BY vs Window Function

| `GROUP BY` | Window Function |
|---|---|
| Collapses rows: one row per group | Keeps **every** row |
| Loses row-level detail | Adds result as a **new column** |
| Example: total salary per department | Example: each employee's salary **and** the department total |

## 3️⃣ Syntax

**PySpark**
```python
from pyspark.sql.window import Window

win = Window.partitionBy("dept").orderBy("salary")
df.withColumn("rank", F.rank().over(win))
```

**SQL**
```sql
RANK() OVER (PARTITION BY dept ORDER BY salary DESC)
```

| Part | Meaning |
|---|---|
| `PARTITION BY` | Splits rows into groups (like GROUP BY, but rows are kept) |
| `ORDER BY` | Order of rows inside each group |
| `ROWS BETWEEN` | *(Optional)* Which rows around the current row to include |

## 4️⃣ Types of Window Functions

| Type | Functions | Used for |
|---|---|---|
| 🏆 **Ranking** | `ROW_NUMBER`, `RANK`, `DENSE_RANK`, `NTILE`, `PERCENT_RANK` | Top N per group, de-duplication |
| 🔍 **Analytic (Value)** | `LAG`, `LEAD`, `FIRST_VALUE`, `LAST_VALUE` | Compare with previous/next row, month-over-month change |
| ➕ **Aggregate** | `SUM`, `AVG`, `MIN`, `MAX`, `COUNT` with `OVER()` | Running totals, moving averages, group totals |

## 5️⃣ 🎤 Interview Question: ROW_NUMBER vs RANK vs DENSE_RANK

Salaries: **5000, 5000, 4000**

| Salary | `ROW_NUMBER` | `RANK` | `DENSE_RANK` |
|---|---|---|---|
| 5000 | 1 | 1 | 1 |
| 5000 | 2 | 1 | 1 |
| 4000 | 3 | **3** | **2** |

- **ROW_NUMBER:** always unique, even for ties
- **RANK:** ties share a rank, then **skips** numbers (gap)
- **DENSE_RANK:** ties share a rank, **no gap**

💡 Use `ROW_NUMBER` to remove duplicates, and `DENSE_RANK` for "Top N distinct salaries".

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col,row_number

In [0]:
data = [(33), (45), (57), (99), (100)]

df = spark.createDataFrame(data, "marks int")

display(df)

In [0]:
df_row = df.withColumn("marks_sequence", row_number().over(Window.orderBy(col("marks").asc())))

display(df_row)

In [0]:
data = [
    ('ABC', 'Math',    33),
    ('DEF', 'Math',    45),
    ('LMN', 'Math',    57),
    ('ABC', 'Science', 99),
    ('DEF', 'Science', 100),
    ('LMN', 'Science', 99)
]

schema= "name String, subject string, marks int"

df = spark.createDataFrame(data, schema)

display(df)

In [0]:
df_row = df.withColumn("marks_sequence", row_number().over(Window.orderBy("marks")))

display(df_row)

In [0]:
from pyspark.sql.functions import rank

df_rank =df.withColumn("marks_rank", rank().over(Window.orderBy("marks")))

display(df_rank)

In [0]:
from pyspark.sql.functions import dense_rank

df_dense_rank = df.withColumn("marks_dense_rank", dense_rank().over(Window.orderBy("marks")))

display(df_dense_rank)

In [0]:
from pyspark.sql.functions import rank, dense_rank

df_rank= df.withColumn("marks_rank", rank().over(Window.orderBy("marks")))\
           .withColumn("marks_dense_rank", dense_rank().over(Window.orderBy("marks")))

display(df_rank)
                                            

###Special Functions

In [0]:
data = [
    (2020, 15000),
    (2021, 16000),
    (2022, 14000),
    (2023, 20000),
    (2024, 17000)
]

schema = "year int, sales int"

df_revenue = spark.createDataFrame(data, schema)

display(df_revenue)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum

df_sales = df_revenue.withColumn("tot_sales", sum("sales").over(Window.orderBy("year")))

display(df_sales)

In [0]:
df_tot_sales =(df_revenue.
               withColumn("tot_sales", 
               sum("sales").over(Window.orderBy("year")
               .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)))
)

display(df_tot_sales)

%md
## 🪟 Window Frames: Running Total vs Grand Total

> **Definition:** A **window frame** decides which rows around the current row are included in the calculation.

| Window spec | Default frame | Result |
|---|---|---|
| `Window.orderBy("year")` | First row → **current row** | 📈 Running total |
| `Window.partitionBy()` *(no orderBy)* | **Whole partition** | 🟰 Grand total on every row |
| `.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)` | First row → **last row** | 🟰 Grand total (even with orderBy) |

🎤 **Interview tip:** *"Adding ORDER BY to a window changes SUM from a total into a running total, because the default frame becomes 'start to current row'."*

In [0]:
df_tot_sales = (df_revenue
                .withColumn("tot_sales",
                sum("sales").over(Window.partitionBy()))
)

display(df_tot_sales)

# 🧠 Window Functions: Practice Challenges
Solve each one in **PySpark → Spark SQL → Standard SQL**.

---

## 🟢 Q1. Second Highest Salary (Company-wide)
Find the employee(s) with the **second highest salary** in the company.

💡 *Hint:* There are ties at the top. Which ranking function handles ties **without gaps**?
✅ *Self-check:* If your answer is **9000**, rethink your ranking function.

---

## 🟢 Q2. Highest Paid Employee per Department
Return the top earner(s) in **each department**. If two employees tie, return both.

⭐ *Bonus:* Change it to return the **second highest** earner in each department.

---

## 🟡 Q3. Department Salary Analysis
For every employee, show:

| Column | Description |
|---|---|
| `dept_avg_salary` | Average salary of their department |
| `diff_from_avg` | `salary - dept_avg_salary` |
| `salary_status` | `"Above Avg"` / `"Below Avg"` / `"Equal"` |

💡 *Hint:* Does this window need an `orderBy`? Remember what it does to the frame.

---

## 🟡 Q4. Year-over-Year Growth (Revenue table) 📈
Using `df_revenue`, add these columns:

| Column | Description |
|---|---|
| `prev_year_sales` | Sales of the previous year → `lag` |
| `next_year_sales` | Sales of the next year → `lead` |
| `yoy_growth_pct` | `(sales - prev_year_sales) / prev_year_sales * 100`, rounded to 2 decimals |
| `trend` | `"Up"` / `"Down"` / `"N/A"` for the first year |

✅ *Self-check:* 2023 growth should be **42.86%**.

⭐ *Bonus:* Instead of `null` for 2020's `prev_year_sales`, show `0`. (Check the **3rd argument** of `lag`.)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

emp_data = [
    (1,  "Alice",   "Sales",       5000),
    (2,  "Bob",     "Sales",       4000),
    (3,  "Charlie", "Sales",       5000),
    (4,  "David",   "Sales",       3000),
    (5,  "Eva",     "Engineering", 9000),
    (6,  "Frank",   "Engineering", 8000),
    (7,  "Grace",   "Engineering", 9000),
    (8,  "Hari",    "Engineering", 7000),
    (9,  "Isha",    "HR",          4500),
    (10, "Ravi",     "HR",          3500),
]

emp_df = spark.createDataFrame(emp_data, "emp_id int, name string, dept string, salary int")
emp_df.createOrReplaceTempView("employees")
df_revenue.createOrReplaceTempView("revenue")

display(emp_df)

In [0]:
df_sal_rank = (emp_df
               .withColumn("sal_rank",
                dense_rank().over(Window.partitionBy("dept")
                                  .orderBy("salary")))
               .filter(col("sal_rank")==2)
)

display(df_sal_rank)